In [3]:
import pandas as pd;
import numpy as np;
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
import scipy.stats as stats

plt.rcParams.update({'font.size': 14})
pd.set_option('display.float_format', lambda x: '%.2f' % x)
path = '../../../../playwright/results/core-web-vitals/testrun-8/'


In [4]:
features = ['navTime', 'totalTime', 'lcp', 'fcp', 'ttfb', 'tbt', 'tti', 'longestTask', 'longTasks', 'nf:init', 'nf:config','nf:loaded']

dirty_dfs =	{
  "Monolith": pd.read_csv(f'{path}2024-12-19T10:13:15.254Z_results-monolith-throttled.csv', sep=',').iloc[5:],
  "CSR": pd.read_csv(f'{path}2024-12-16T11:08:37.924Z_results-csr-throttled.csv', sep=',').iloc[5:],
  "CSR sd": pd.read_csv(f'{path}2024-12-16T11:08:37.924Z_results-csr-throttled.csv', sep=',').iloc[5:],
  "SSRH": pd.read_csv(f'{path}2024-12-15T23:30:49.336Z_results-ssrh-throttled.csv', sep=',').iloc[5:],
  "SSRH sd": pd.read_csv(f'{path}2024-12-17T12:56:03.620Z_results-ssrh-sd-throttled.csv', sep=',').iloc[5:],
  "SSRV": pd.read_csv(f'{path}2024-12-17T16:46:34.776Z_results-ssrv-sd.csv', sep=',').iloc[5:],
}

In [5]:
def detect_outliers(_df, _features, contamination=0.1):
    clf = IsolationForest(contamination=contamination, random_state=42)
    outliers = clf.fit_predict(_df[_features])
    return outliers == 1

masks = {}
dfs = {}
target_features = ['navTime', 'totalTime', 'lcp', 'fcp', 'ttfb']

for name, _df in dirty_dfs.items():
    mask = detect_outliers(_df, target_features)
    masks[name] = mask
    dfs[name] = _df[mask].copy()

In [6]:
columns = [ 'ttfb','fcp','nf:init','lcp','tti','nf:loaded','tbt','longestTask']
rows = []

for name, df in dfs.items():
    mean_row = df[columns].mean()
    rows.append((f"{name} (mean)", mean_row))

for name, df in dfs.items():
    percentile_row = df[columns].quantile(0.75)
    rows.append((f"{name} (75th)", percentile_row))

result_df = pd.DataFrame([row[1] for row in rows], index=[row[0] for row in rows])
result_df = result_df.mask(result_df < 0, '-')
result_df

,ttfb,fcp,nf:init,lcp,tti,nf:loaded,tbt,longestTask
Monolith (mean),354.21,1144.63,-,1144.63,1144.63,-,0.00,-
CSR (mean),340.01,1131.01,1341.10,4543.92,3776.53,3819.91,34.79,84.79
CSR sd (mean),340.01,1131.01,1341.10,4543.92,3776.53,3819.91,34.79,84.79
SSRH (mean),369.79,1189.97,1482.76,1189.97,3510.37,3719.97,16.09,66.09
SSRH sd (mean),369.62,1190.44,1482.99,1190.44,5283.62,5297.43,31.26,81.26
SSRV (mean),21.32,86.11,-,86.17,86.11,-,0.00,-
Monolith (75th),354.80,1151.50,-,1151.50,1151.50,-,0.00,-
CSR (75th),340.30,1135.90,1345.80,4553.88,3783.17,3825.98,35.00,85.00
CSR sd (75th),340.30,1135.90,1345.80,4553.88,3783.17,3825.98,35.00,85.00
SSRH (75th),370.60,1193.90,1487.50,1193.90,3515.20,3723.40,17.00,67.00
